# squeeze — shrink a blob before it eats the context window

A verbose log dump is mostly repetition. `compress()` shrinks it toward a token target and hands back a **reversible** handle, so you send 400 tokens and can still restore the original byte-for-byte.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · 1,500 lines of repetitive logs

In [ ]:
import main as recipe
from cendor.core import tokens

logs = recipe.noisy_logs()
print(logs[:220], "…")
print(f"\n{tokens.count(logs, 'gpt-4o'):,} tokens, {len(logs) / 1024:.1f} KB")

## 2 · Compress toward a target

`kind="auto"` detects the shape (json / logs / code / prose) and picks the technique.

In [ ]:
from cendor.squeeze import compress

small, handle = compress(logs, kind="auto", target_tokens=400)
print(f"kind      : {handle.kind}  (technique: {handle.technique})")
print(f"tokens    : {tokens.count(logs, 'gpt-4o'):,} -> {tokens.count(small, 'gpt-4o'):,}")

## 3 · Reversible, not lossy-and-hope

The handle carries what it needs to reconstruct the original exactly. That is the difference between compression and summarisation.

In [ ]:
restored = handle.expand()
print(f"byte-for-byte identical: {restored == logs}")

## 4 · Prove it

In [ ]:
assert restored == logs, "expand() must restore the original exactly"
assert tokens.count(small, "gpt-4o") <= 400, "must respect the token target"
print("OK")